### Installation

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab and Kaggle notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
    !pip install --no-deps cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
%pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --quiet


### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Choose any! We auto support RoPE Scaling internally!
dtype = (
    None  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
)

load_in_8bit = True  # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",  # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",  # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit",  # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",  # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",  # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",  # Gemma 2x faster!
]  # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
    load_in_8bit=False,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 128,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

<a name="Data"></a>
### Data Prep
We now use the Alpaca dataset from [yahma](https://huggingface.co/datasets/yahma/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/drive/1XamvWYinY6FOSX9GLvnqSjjsNflxdhNc?usp=sharing).

For text completions like novel writing, try this [notebook](https://colab.research.google.com/drive/1ef-tab5bhkvWmBOObepl1WgJvfvSzn5Q?usp=sharing).

In [ ]:
# ShareGPT format version
sharegpt_prompt = """<|im_start|>user
{}
{}
<|im_end|>
<|im_start|>assistant
{}
<|im_end|>"""

# For your formatting function
def formatting_prompts_func(examples):
    instructions = examples["question"]
    inputs = examples["context"]
    outputs = examples["answers"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Combine instruction and input for the user message
        user_message = instruction
        if input and input.strip():  # Add context only if it exists and is not empty
            user_message += f"\n\nContext: {input}"

        answer_text = output['text'][0] if output.get('text', []) else ''
        text = sharegpt_prompt.format(user_message, "", answer_text) + tokenizer.eos_token
        texts.append(text)
    return {"text": texts}

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

# **Wandb Integration**


In [ ]:
# Install library
!pip install wandb --upgrade --quiet

# Setting up Wandb
!wandb login <>

import os

os.environ["WANDB_PROJECT"] = "<DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit>"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    # formatting_func = formatting_prompts_func,
    train_dataset = dataset,
    # eval_dataset = dataset_test,
    tokenizer = tokenizer,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    logging_steps = 1,
    dataset_num_proc = 4,  # Increased for faster processing
    # packing = True,  # Enable packing for more efficient training
    args = TrainingArguments(
        per_device_train_batch_size = 2,  # Reduced to handle larger model
        gradient_accumulation_steps = 16,  # Increased to compensate for smaller batch size
        warmup_steps = 5,  # More warmup steps for better stability
        max_steps = 40,  # Increased training steps
        learning_rate = 2e-4,  # Lower learning rate for better convergence
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        optim = "adamw_8bit",  # Use 8-bit Adam for memory efficiency
        weight_decay = 0.05,  # Increased weight decay to prevent overfitting
        lr_scheduler_type = "cosine",  # Cosine scheduler for better convergence
        seed = 42,
        output_dir = "gemma2_persian_qa_model",
        save_strategy = "steps",
        save_steps = 3 ,# Change if needed
        # eval_strategy = "steps",
        # eval_steps = 100,
        # load_best_model_at_end = True,
        metric_for_best_model = "accuracy",
        greater_is_better = True,
        report_to = "wandb",
        logging_steps = 1, # Change if needed

    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

# **Rsuming Wanddb From Last Checkpoint**

In [ ]:
import wandb
import torch
run = wandb.init(name = "sparkling-sponge-7")
artifact = run.use_artifact('m-akhoundi22-boston/<DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit>/model-ezfi1ozv:v15', type='model')
artifact_dir = artifact.download()
trainer.train(resume_from_checkpoint=artifact_dir)

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "fdsfsdfdsf؟", # instruction
        "ddsdds؟", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "زمان چیه؟", # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

In [ ]:
from huggingface_hub import login #writing to huggingface
login(token="")

In [ ]:
model.push_to_hub("regd/gemma29b1000step5e-5")
trainer.create_model_card("regd/gemma29b1000step5e-5")
tokenizer.push_to_hub("regd/gemma29b1000step5e-5")

In [ ]:
from huggingface_hub import login #reading from huggingface
login(token="")

In [ ]:
%pip install -U bitsandbytes --quiet
%pip install datasets --quiet
%pip install tqdm --quiet
%pip install bert_score --quiet


## **Model Evaluation**

In [ ]:
# dataset_test
def format_prompt(question, context=None):
         if context:
             prompt = f"Context: {context}\n\nQuestion: {question}\n\nAnswer:"
         else:
             prompt = f"Question: {question}\n\nAnswer:"
         return prompt

In [ ]:
def generate_answer(model, tokenizer, prompt, max_new_tokens=20, max_seq_length=2048):
         inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
         with torch.no_grad():
             outputs = model.generate(
                 **inputs,
                 max_new_tokens=max_new_tokens,
                 temperature=0.7, # Adjust as needed
                 top_p=0.9,      # Adjust as needed
                 do_sample=False,
                 max_length = 20,
                 eos_token_id=tokenizer.eos_token_id,
                 use_cache=False,
                #  num_beams=5,
                #  early_stopping=True
             )
         answer = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
         return answer.strip()



In [ ]:
def evaluate_f1_token(prediction, reference):
          pred_tokens = prediction.lower().split()
          ref_tokens = reference.lower().split()
          common = set(pred_tokens) & set(ref_tokens)
          if len(pred_tokens) == 0 or len(ref_tokens) == 0:
              return 0.0
          precision = len(common) / len(pred_tokens)
          recall = len(common) / len(ref_tokens)
          if precision + recall == 0:
              return 0.0
          f1 = 2 * precision * recall / (precision + recall)
          return f1

In [ ]:
from bert_score import score

def evaluate_bert_score(prediction, reference, lang="en", model_type=None, num_layers=None, verbose=False):
    """
    Calculates BERTScore for a single prediction and reference.

    Args:
        prediction (str): The predicted sentence.
        reference (str): The reference sentence.
        lang (str): Language of the sentences (e.g., "en", "es").
        model_type (str, optional): BERT model to use (e.g., "bert-base-uncased"). If None, it will be automatically inferred from lang.
        num_layers (int, optional): Number of BERT layers to use.
        verbose (bool, optional): Print progress.

    Returns:
        float: The F1-score from BERTScore, or 0.0 if there is an error.
    """
    try:
        P, R, F1 = score([prediction], [reference], lang=lang, model_type=model_type, num_layers=num_layers, verbose=verbose)
        return F1.item()  # Return the F1 score as a float
    except Exception as e:
        print(f"Error calculating BERTScore: {e}")
        return 0.0

# Example usage (replace your existing evaluate_f1_token calls):
# ... (your existing code) ...

# Replace calls to evaluate_f1_token with evaluate_bert_score
# f1_score = evaluate_f1_token(prediction, reference)
# f1_score = evaluate_bert_score(prediction, reference)

# ... (rest of your code) ...

In [ ]:
def evaluate_exact_match(prediction, reference):
    """
    Calculates the Exact Match (EM) score between a prediction and a reference.

    Args:
        prediction (str): The predicted answer.
        reference (str): The reference answer.

    Returns:
        float: 1.0 if the prediction exactly matches the reference, 0.0 otherwise.
    """
    if prediction.strip().lower() == reference.strip().lower():
        return 1.0
    else:
        return 0.0

In [ ]:
import torch
from tqdm import tqdm
from datasets import load_dataset
dataset_test = load_dataset("SajjadAyoubi/persian_qa", split = "validation")
dataset_test = dataset_test.select(range(20))
results = []
for item in tqdm(dataset_test, desc="Evaluating"):
      question = item["question"]
      if "answers" in item and "text" in item["answers"] and len(item["answers"]["text"]) > 0:
        reference = item["answers"]["text"][0]  # Assuming a list of answers
      else:
          reference = ""
      context = item.get("context", None)
      prompt = format_prompt(question, context)
      prediction = generate_answer(model, tokenizer, prompt)
      f1_score = evaluate_f1_token(prediction, reference)
      em_score = evaluate_exact_match(prediction, reference)
      bert_score = evaluate_bert_score(prediction, reference)
      results.append({
        "question": question,
        "reference": reference,
        "prediction": prediction,
        "f1_score": f1_score,
        "exact_match": em_score,
        "bert_score": bert_score
      })
results



In [ ]:
import numpy as np
avg_f1 = np.mean([result["f1_score"] for result in results])
avg_em = np.mean([result["exact_match"] for result in results])
avg_bert = np.mean([result["bert_score"] for result in results])
print(f"Average F1 Score: {avg_f1:.4f}")
print(f"Average Exact Match: {avg_em:.4f}")
print(f"Average BERT Score: {avg_bert:.4f}")

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="lora_model",  # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
    FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
    [
        alpaca_prompt.format(
            "What is a famous tall tower in Paris?",  # instruction
            "",  # input
            "",  # output - leave this blank for generation!
        )
    ],
    return_tensors="pt",
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
tokenizer.batch_decode(outputs)

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "", # Get a token at https://huggingface.co/settings/tokens
    )